# Machine Learning-Based Buyer Segmentation and Investment Profiling for Real Estate Market Intelligence

**Dataset note:** The bundled dataset is synthetic practice data, not actual Parcl customer data. Results are illustrative only.


In [ ]:
import pandas as pd, numpy as np
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.cluster import KMeans, AgglomerativeClustering
from sklearn.metrics import silhouette_score

# Load the original CSV from the repository root.
df = pd.read_csv('real_estate_buyer_synthetic_dataset.csv')
print('Raw dataset shape:', df.shape)
display(df.head())


In [ ]:
print('Missing values before cleaning:')
display(df.isna().sum())
print('Exact duplicate rows:', df.duplicated().sum())

# Remove exact duplicates and duplicate client IDs.
df = df.drop_duplicates().drop_duplicates(subset=['client_id'], keep='first').copy()
print('After duplicate removal:', df.shape)

# Match the 594-record dataset used by the live dashboard:
# exclude records where region is missing.
df = df.dropna(subset=['region']).copy()

# Convert fields used in analysis.
df['date_of_birth'] = pd.to_datetime(df['date_of_birth'], errors='coerce')
df['age'] = ((pd.Timestamp('2026-11-11') - df['date_of_birth']).dt.days / 365.25).round()
df.loc[~df['age'].between(18,100), 'age'] = np.nan
df['satisfaction_score'] = pd.to_numeric(df['satisfaction_score'], errors='coerce')

print('Final dashboard-aligned shape:', df.shape)
assert len(df) == 594, f'Expected 594 rows, found {len(df)}'
assert df['client_id'].nunique() == 594, 'Client IDs are not unique'


In [ ]:
cat = ['client_type','gender','country','region','acquisition_purpose','loan_applied','referral_channel']
num = ['age','satisfaction_score']

# Do not use an existing cluster column as an input feature.
feature_cols = cat + num
prep = ColumnTransformer([
    ('num', Pipeline([
        ('imputer', SimpleImputer(strategy='median')),
        ('scale', StandardScaler())
    ]), num),
    ('cat', Pipeline([
        ('imputer', SimpleImputer(strategy='most_frequent')),
        ('encode', OneHotEncoder(handle_unknown='ignore'))
    ]), cat)
])
X = prep.fit_transform(df[feature_cols])
display(df.groupby('acquisition_purpose').agg(
    buyers=('client_id','count'),
    satisfaction=('satisfaction_score','mean')
))


In [ ]:
rows=[]
for k in range(2,8):
    model=KMeans(n_clusters=k,random_state=42,n_init=10)
    labels=model.fit_predict(X)
    rows.append({'k':k,'inertia':model.inertia_,'silhouette':silhouette_score(X,labels)})
evaluation=pd.DataFrame(rows)
display(evaluation)
best_k=int(evaluation.loc[evaluation.silhouette.idxmax(),'k'])
df['kmeans_cluster']=KMeans(n_clusters=best_k,random_state=42,n_init=10).fit_predict(X)
df['hierarchical_cluster']=AgglomerativeClustering(n_clusters=best_k).fit_predict(X.toarray() if hasattr(X,'toarray') else X)
display(df.groupby('kmeans_cluster').agg(buyers=('client_id','count'), median_age=('age','median'), investment_share=('acquisition_purpose',lambda s:(s=='Investment').mean()), loan_share=('loan_applied',lambda s:(s=='Yes').mean()), satisfaction=('satisfaction_score','mean')))


## Reproducibility note
This notebook loads `real_estate_buyer_synthetic_dataset.csv` from the repository root, removes exact duplicate rows and duplicate client IDs, then excludes rows with missing region to match the 594-record dashboard dataset. Other missing feature values are imputed in the preprocessing pipeline. Run all cells from top to bottom and use the freshly generated metrics and profiles.